In [1]:
import math
import pickle
import random

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize

from agent import FloodWarningEnv
from baselines.random_baseline import RandomAgent
from baselines.threshold_baseline import ThresholdAgent

import matplotlib.pyplot as plt

### Evaluation

In [ ]:
SEASON_TO_IDX = {"Spring": 0, "Summer": 1, "Autumn": 2, "Winter": 3}

SEEDS = [2, 3, 4]#, 5, 6] # 5 seeds, exclude seed 1 since it was used for training the RL agents
N_EPISODES = 5000
FALSE_WEIGHT = 1.5
MISSED_WEIGHT = 2
JUMP_WEIGHT = 0.5
RESULTS_PATH = "evaluation_results.csv"

LSTM_MODEL_PATH = "models/lstm/best_model"
VECNORM_MODEL_PATH = "models/lstm_vecnorm/best_model"
VECNORM_STATS_PATH = "models/lstm_vecnorm/vec_normalize.pkl"

def make_env():
    return lambda: FloodWarningEnv(severe_prob=0.0)

def get_underlying_env(vec_env):
    e = vec_env.venv if isinstance(vec_env, VecNormalize) else vec_env
    return e.envs[0].env

def get_threshold_obs(raw_env):
    f = raw_env.env.get_observable_features()
    idx = SEASON_TO_IDX[f["season"]]
    return {
        **f,
        "season_sin": math.sin(2 * math.pi * idx / 4),
        "season_cos": math.cos(2 * math.pi * idx / 4),
    }

def compute_reward(action, impact_level, prev_action):
    alignment = 1.0 if action == impact_level else 0.0
    false_alarm = FALSE_WEIGHT * action * max(0, action - impact_level)
    missed = MISSED_WEIGHT * impact_level * max(0, impact_level - action)
    jump = JUMP_WEIGHT * max(0, (action - prev_action) - 1)
    return alignment - false_alarm - missed - jump

def run_raw_episode(env, agent_fn):
    obs, _ = env.reset()
    steps, prev_action, done = [], 0, False
    while not done:
        action = agent_fn(obs, env)
        obs, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        impact = env._get_impact_level(env.env.impact)
        steps.append({
            "action": int(action),
            "impact_level": impact,
            "impact_score": float(env.env.impact),
            "reward": compute_reward(action, impact, prev_action),
            "correct": action == impact,
            "false_alarm": action > impact,
            "missed": action < impact,
            "level_diff": action - impact,
        })
        prev_action = action
    return steps

def run_ppo_episode(vec_env, model):
    obs = vec_env.reset()
    lstm_states = None
    ep_starts = np.ones((1,), dtype=bool)
    steps, prev_action, done = [], 0, False
    raw = get_underlying_env(vec_env)
    while not done:
        action, lstm_states = model.predict(
            obs, state=lstm_states, episode_start=ep_starts, deterministic=True
        )
        obs, _, dones, _ = vec_env.step(action)
        done = dones[0]
        ep_starts = dones
        impact = raw._get_impact_level(raw.env.impact)
        a = int(action[0])
        steps.append({
            "action": a,
            "impact_level": impact,
            "impact_score": float(raw.env.impact),
            "reward": compute_reward(a, impact, prev_action),
            "correct": a == impact,
            "false_alarm": a > impact,
            "missed": a < impact,
            "level_diff": a - impact,
        })
        prev_action = a
    return steps


# Load agents once — they are stateless between seeds
random_agent = RandomAgent()

with open("baselines/dt_model.pkl", "rb") as f:
    clf = pickle.load(f)
threshold_agent = ThresholdAgent(clf)

lstm_env = make_vec_env(make_env(), n_envs=1, seed=SEEDS[0])
lstm_model = RecurrentPPO.load(LSTM_MODEL_PATH, env=lstm_env)

vecnorm_env = make_vec_env(make_env(), n_envs=1, seed=SEEDS[0])
vecnorm_env = VecNormalize.load(VECNORM_STATS_PATH, vecnorm_env)
vecnorm_env.training = False
vecnorm_env.norm_reward = False
vecnorm_model = RecurrentPPO.load(VECNORM_MODEL_PATH, env=vecnorm_env)

raw_env_random = FloodWarningEnv(false_weight=FALSE_WEIGHT, missed_weight=MISSED_WEIGHT, jump_weight=JUMP_WEIGHT, severe_prob=0.0)
raw_env_threshold = FloodWarningEnv(false_weight=FALSE_WEIGHT, missed_weight=MISSED_WEIGHT, jump_weight=JUMP_WEIGHT, severe_prob=0.0)

print("All agents loaded.")

# Evaluation loop
rows = []

for seed in SEEDS:
    print(f"\nSeed {seed}")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    rng = np.random.RandomState(seed)
    episode_seeds = rng.randint(0, 2**31, N_EPISODES)

    for ep in tqdm(range(N_EPISODES), desc=f"Seed {seed}"):
        ep_seed = int(episode_seeds[ep])

        np.random.seed(ep_seed)
        for step_idx, step in enumerate(run_raw_episode(raw_env_random, lambda obs, env: random_agent.predict())):
            rows.append({"agent": "random", "seed": seed, "episode": ep, "step": step_idx, **step})

        np.random.seed(ep_seed)
        for step_idx, step in enumerate(run_raw_episode(raw_env_threshold, lambda obs, env: threshold_agent.predict(get_threshold_obs(env)))):
            rows.append({"agent": "threshold", "seed": seed, "episode": ep, "step": step_idx, **step})

        np.random.seed(ep_seed)
        for step_idx, step in enumerate(run_ppo_episode(lstm_env, lstm_model)):
            rows.append({"agent": "ppo", "seed": seed, "episode": ep, "step": step_idx, **step})

        np.random.seed(ep_seed)
        for step_idx, step in enumerate(run_ppo_episode(vecnorm_env, vecnorm_model)):
            rows.append({"agent": "ppo_vecnorm", "seed": seed, "episode": ep, "step": step_idx, **step})

df = pd.DataFrame(rows)
df.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved to {RESULTS_PATH}")

All agents loaded.

Seed 2


Seed 2: 100%|██████████| 5000/5000 [45:48<00:00,  1.82it/s] 



Seed 3


Seed 3:  97%|█████████▋| 4866/5000 [43:41<01:18,  1.72it/s]

### Visualisations

In [ ]:
# Learning curves during training
models = {
    "lstm": "models/lstm/evaluations.npz",
    "lstm_vecnorm": "models/lstm_vecnorm/evaluations.npz",
}

fig, ax = plt.subplots(figsize=(10, 5))

for name, path in models.items():
    data = np.load(path)
    timesteps = data["timesteps"]
    results = data["results"] 

    mean = results.mean(axis=1)
    std = results.std(axis=1)

    ax.plot(timesteps, mean, label=name)
    ax.fill_between(timesteps, mean - std, mean + std, alpha=0.2)

ax.set_xlabel("Timesteps")
ax.set_ylabel("Mean eval reward")
ax.set_title("Learning curves")
ax.legend()
plt.tight_layout()
plt.savefig("results/learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df = pd.read_csv("evaluation_results.csv")
agents = df["agent"].unique()
LEVELS = [0, 1, 2, 3]
LEVEL_NAMES = ["None", "Yellow", "Amber", "Red"]
COLORS = {"random": "#5B8DB8", "threshold": "#B85B5B", "ppo": "#5BB87A", "ppo_vecnorm": "#8B5BB8"}

# False alarm vs missed
fa = df.groupby("agent")["false_alarm"].mean()
ms = df.groupby("agent")["missed"].mean()

x = np.arange(len(agents))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, [fa[a] for a in agents], width, label="False alarm", color="tomato")
ax.bar(x + width/2, [ms[a] for a in agents], width, label="Missed warning", color="steelblue")
ax.set_xticks(x); ax.set_xticklabels(agents)
ax.set_ylabel("Rate")
ax.set_title("False alarm vs missed warning rate")
ax.legend()
plt.tight_layout()
plt.savefig("results/eval_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

# Warning level distribution
fig, axes = plt.subplots(1, len(agents) + 1, figsize=(4 * (len(agents) + 1), 4))

true_dist = df[df["agent"] == agents[0]]["impact_level"].value_counts(normalize=True).sort_index()
axes[0].bar(LEVEL_NAMES, [true_dist.get(l, 0) for l in LEVELS], color="gray")
axes[0].set_title("True distribution"); axes[0].set_ylabel("Proportion")

for ax, agent in zip(axes[1:], agents):
    sub = df[df["agent"] == agent]
    dist = sub["action"].value_counts(normalize=True).sort_index()
    ax.bar(LEVEL_NAMES, [dist.get(l, 0) for l in LEVELS], color=COLORS[agent])
    ax.set_title(agent)

plt.suptitle("Warning distribution vs true impact distribution")
plt.tight_layout()
plt.savefig("results/eval_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Reward distribution
ep_rewards = df.groupby(["agent", "episode"])["reward"].sum().reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
bp = ax.boxplot(
    [ep_rewards[ep_rewards["agent"] == a]["reward"].values for a in agents],
    labels=agents, patch_artist=True,
)
for patch, agent in zip(bp["boxes"], agents):
    patch.set_facecolor(COLORS[agent])
    patch.set_alpha(0.6)

ax.set_ylabel("Episode reward")
ax.set_title("Reward distribution across episodes")
plt.tight_layout()
plt.savefig("results/eval_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

# Impact score vs reward scatter (by step)
ACTION_COLORS = {0: "gray", 1: "yellow", 2: "orange", 3: "red"}
LABELS = {0: "None", 1: "Yellow", 2: "Amber", 3: "Red"}
IMPACT_THRESHOLDS = {"none": 0.0447, "yellow": 0.0673, "amber": 0.1657}
THRESHOLD_STYLES = {
    "none":   {"color": "gray",   "linestyle": "dotted"},
    "yellow": {"color": "gold",   "linestyle": "dashed"},
    "amber":  {"color": "orange", "linestyle": "dashdot"},
}

fig, axes = plt.subplots(1, len(agents), figsize=(5 * len(agents), 5))

for ax, agent in zip(axes, agents):
    sub = df[df["agent"] == agent]

    for action in LEVELS:
        subset = sub[sub["action"] == action]
        ax.scatter(subset["step"], subset["impact_score"],
                   c=ACTION_COLORS[action], label=LABELS[action], alpha=0.3, s=5)

    x_max = df["step"].max()
    for label, value in IMPACT_THRESHOLDS.items():
        style = THRESHOLD_STYLES[label]
        ax.axhline(y=value, color=style["color"], linestyle=style["linestyle"],
                   linewidth=1.5, alpha=0.5, zorder=3)
        ax.text(x_max, value, f" {label}", color=style["color"],
                va="center", ha="left", fontsize=8, alpha=0.9)

    ax.set_ylim(df["impact_score"].min(), df["impact_score"].max())
    ax.set_xlabel("Step")
    ax.set_ylabel("Impact Score")
    ax.set_title(f"{agent} agent")
    ax.legend(markerscale=2)
    ax.grid(alpha=0.2)

plt.suptitle("Impact Score by Issued Warning Level (Episode Steps)", fontsize=14)
plt.tight_layout()
plt.savefig("results/eval_impact_vs_action.png", dpi=150, bbox_inches="tight")
plt.show()

# Impact score vs reward scatter (sequential)
fig, axes = plt.subplots(1, len(agents), figsize=(5 * len(agents), 5))

for ax, agent in zip(axes, agents):
    sub = df[df["agent"] == agent].copy()
    sub["global_step"] = sub["episode"] * 200 + sub["step"]

    for action in LEVELS:
        subset = sub[sub["action"] == action]
        ax.scatter(subset["global_step"], subset["impact_score"],
                   c=ACTION_COLORS[action], label=LABELS[action], alpha=0.3, s=5)

    x_max = sub["global_step"].max()
    for label, value in IMPACT_THRESHOLDS.items():
        style = THRESHOLD_STYLES[label]
        ax.axhline(y=value, color=style["color"], linestyle=style["linestyle"],
                   linewidth=1.5, alpha=0.5, zorder=3)
        ax.text(x_max, value, f" {label}", color=style["color"],
                va="center", ha="left", fontsize=8, alpha=0.9)

    ax.set_ylim(df["impact_score"].min(), df["impact_score"].max())
    ax.set_xlabel("Global step")
    ax.set_ylabel("Impact Score")
    ax.set_title(f"{agent} agent")
    ax.legend(markerscale=2)
    ax.grid(alpha=0.2)
    ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
    ax.set_xlabel("Global step")    

plt.suptitle("Impact Score by Issued Warning Level (Global Steps)", fontsize=14)
plt.tight_layout()
plt.savefig("results/eval_impact_vs_action_sequential.png", dpi=150, bbox_inches="tight")
plt.show()

# Mean reward per step
fig, ax = plt.subplots(figsize=(10, 4))
for agent in agents:
    sub = df[df["agent"] == agent]
    mean_by_step = sub.groupby("step")["reward"].mean()
    ax.plot(mean_by_step.index, mean_by_step.values, label=agent, color=COLORS[agent])
ax.set_xlabel("Step")
ax.set_ylabel("Mean reward")
ax.set_title("Mean reward per episode step")
ax.legend()
plt.tight_layout()
plt.savefig("results/eval_reward_by_step.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
seeds = df["seed"].unique()

# False alarm vs missed — with error bars across seeds
fa = df.groupby(["agent", "seed"])["false_alarm"].mean().groupby("agent")
ms = df.groupby(["agent", "seed"])["missed"].mean().groupby("agent")

x = np.arange(len(agents))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, fa.mean(), width, yerr=fa.std(), capsize=4, label="False alarm", color="tomato")
ax.bar(x + width/2, ms.mean(), width, yerr=ms.std(), capsize=4, label="Missed warning", color="steelblue")
ax.set_xticks(x); ax.set_xticklabels(agents)
ax.set_ylabel("Rate")
ax.set_title("False alarm vs missed warning rate (mean ± std across seeds)")
ax.legend()
plt.tight_layout()
plt.savefig("results/eval_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

# Reward boxplot — grouped by agent and seed
ep_rewards = df.groupby(["agent", "seed", "episode"])["reward"].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
positions, data, ticks, tick_labels = [], [], [], []
for i, agent in enumerate(agents):
    for j, seed in enumerate(seeds):
        sub = ep_rewards[(ep_rewards["agent"] == agent) & (ep_rewards["seed"] == seed)]
        pos = i * (len(seeds) + 1) + j
        positions.append(pos)
        data.append(sub["reward"].values)
    ticks.append(i * (len(seeds) + 1) + (len(seeds) - 1) / 2)
    tick_labels.append(agent)

bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
for patch, pos in zip(bp["boxes"], positions):
    agent_idx = pos // (len(seeds) + 1)
    patch.set_facecolor(list(COLORS.values())[agent_idx])
    patch.set_alpha(0.6)

ax.set_xticks(ticks); ax.set_xticklabels(tick_labels)
ax.set_ylabel("Episode reward")
ax.set_title("Reward distribution per agent per seed")
plt.tight_layout()
plt.savefig("results/eval_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

# Mean reward per seed — shows cross-seed consistency
seed_means = ep_rewards.groupby(["agent", "seed"])["reward"].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(agents))
width = 0.25
for j, seed in enumerate(seeds):
    vals = [seed_means[(seed_means["agent"] == a) & (seed_means["seed"] == seed)]["reward"].values[0] for a in agents]
    ax.bar(x + j * width, vals, width, label=f"Seed {seed}", alpha=0.8)
ax.set_xticks(x + width); ax.set_xticklabels(agents)
ax.set_ylabel("Mean episode reward")
ax.set_title("Mean reward per agent per seed")
ax.legend()
plt.tight_layout()
plt.savefig("results/eval_reward_per_seed.png", dpi=150, bbox_inches="tight")
plt.show()

# Mean reward per step — averaged across seeds with shaded std
fig, ax = plt.subplots(figsize=(10, 4))
for agent in agents:
    sub = df[df["agent"] == agent]
    step_seed = sub.groupby(["seed", "step"])["reward"].mean()
    mean_by_step = step_seed.groupby("step").mean()
    std_by_step = step_seed.groupby("step").std()
    ax.plot(mean_by_step.index, mean_by_step.values, label=agent, color=COLORS[agent])
    ax.fill_between(mean_by_step.index,
                    mean_by_step - std_by_step,
                    mean_by_step + std_by_step,
                    alpha=0.15, color=COLORS[agent])
ax.set_xlabel("Step")
ax.set_ylabel("Mean reward")
ax.set_title("Mean reward per episode step (shaded = std across seeds)")
ax.legend()
plt.tight_layout()
plt.savefig("results/eval_reward_by_step.png", dpi=150, bbox_inches="tight")
plt.show()